# Pannocchia CLQR: Custom Loss Combinations

Train non-uniform timesteps for the Pannocchia constrained LQR system using a fully configurable composite loss:
$$\mathcal{L} = w_{\text{ocp}} \cdot \mathcal{L}_{\text{ocp}} + \sum_i w_i \cdot \mathcal{L}_i$$

Set `loss_weights` to whichever regularizers you want to combine, with their fixed weights. Available losses: `L_IV`, `L_EQ`, `L_CPC`, `L_CSS`, `L_defect`, `L_dyn`, `L_equi`, `L_FI`, `L_SC`, `L_PWLH`, `L_SSD`.

In [ ]:
import sys, os
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'src'))
from dimp.utils import init_matplotlib, get_colors
init_matplotlib()
colors = get_colors()

from pann_train import train_custom_loss
from pann_prob import T
from utils import plot_loss_results

## Configuration

Edit `loss_weights` and the training settings below to test different combinations.

In [ ]:
ocp_weight = 1.0                     # weight on the OCP (task) loss L_ocp
loss_weights = {"L_IV_sym": 0.0}     # {regularizer_name: weight}; see registry in cell above
discretization = "zoh"               # dynamics+cost discretization: "zoh" (exact) or "euler"
detach = "none"                      # gradient detach mode: "none" | "reg" (stop grad through regs) | "all"
n_steps = 40                         # number of timesteps in the horizon
n_epochs = 500                      # number of training epochs
lr = 5e-2                            # Adam learning rate on theta (softmax logits for dt)

# Softmax temperature schedule for theta -> dt: None = constant tau=1,
# else (kind, tau_start, tau_end) with kind in {"linear", "exp"}; only a *changing* tau matters.
tau_schedule = None #("exp", 5.0, 0.5)

# r-adaptivity: discrete merge/split moves on the time grid, accepted via Metropolis
# (simulated annealing). Periodically picks an interval to split and a neighbor pair
# to merge, accepting if loss drops or with prob exp(-dL/T_metro).
radapt_enable = True                                  # master switch
radapt_every = 5                                      # attempt a move every K epochs
radapt_freq_schedule = None                           # or (start_every, end_every) to anneal frequency
radapt_importance = "control_var"                     # interval-importance score: "cost_density" | "control_var" | "combined"
radapt_beta = 1000                                    # inverse-temp for picking which intervals to split/merge
radapt_temp_schedule = ("exp", 1e-4, 1e-8)            # Metropolis temperature schedule (kind, T_start, T_end)
radapt_warmup = 50                                    # skip moves before this epoch
radapt_cooldown = 50                                  # skip moves in the last K epochs
radapt_seed = 0                                       # RNG seed for move selection (separate from torch)

## Train

In [ ]:
sol_custom, history_custom, n_custom = train_custom_loss(
    loss_weights, n=n_steps, n_epochs=n_epochs, lr=lr,
    discretization=discretization, ocp_weight=ocp_weight,
    detach=detach, tau_schedule=tau_schedule,
    radapt_enable=radapt_enable,
    radapt_every=radapt_every,
    radapt_freq_schedule=radapt_freq_schedule,
    radapt_importance=radapt_importance,
    radapt_beta=radapt_beta,
    radapt_temp_schedule=radapt_temp_schedule,
    radapt_warmup=radapt_warmup,
    radapt_cooldown=radapt_cooldown,
    radapt_seed=radapt_seed,
)


## Plot results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)

epochs = [h["epoch"] for h in history_custom]

axes[0].plot(epochs, [h["loss"] for h in history_custom], label="total")
axes[0].plot(epochs, [ocp_weight * h["loss_ocp"] for h in history_custom],
             label=f"{ocp_weight}·L_ocp", ls="--")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss curves")
axes[0].legend(fontsize=7)

for name, w in loss_weights.items():
    axes[1].plot(epochs, [w * h[f"loss_{name}"] for h in history_custom],
                 label=f"{w}·{name}")
axes[1].plot(epochs, [h["loss_reg_total"] for h in history_custom],
             label="reg total", ls="--", color="k")
axes[1].set(xlabel="Epoch", ylabel="Loss", title="Weighted regularizer losses")
axes[1].legend(fontsize=7)

dts_final = np.array(history_custom[-1]["dts"]).flatten()
times_final = np.concatenate([[0], np.cumsum(dts_final)])
axes[2].step(times_final[:-1], dts_final, where="post")
axes[2].axhline(T / n_custom, color="gray", ls=":", label="uniform dt")
axes[2].set(xlabel="Time", ylabel="dt", title="Learned timesteps")
axes[2].legend(fontsize=7)

label = f"{ocp_weight}·L_ocp + " + " + ".join(f"{w}·{n}" for n, w in loss_weights.items())
fig.suptitle(f"Custom: {label}", fontsize=11)
plt.show()

In [ ]:
result_custom = {"sol": sol_custom, "history": history_custom, "n": n_custom}
plot_loss_results(label, result_custom, results_dir=None, show=True)

In [ ]:
attempts = [h for h in history_custom if h["radapt_attempted"]]
if attempts:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), constrained_layout=True)

    eps_attempt = np.array([h["epoch"] for h in attempts])
    dLs = np.array([h["radapt_dL"] for h in attempts])
    accepted = np.array([h["radapt_accepted"] for h in attempts])

    sc = axes[0].scatter(eps_attempt[accepted], dLs[accepted], c=dLs[accepted],
                         cmap="viridis", marker="o", label="accepted")
    axes[0].scatter(eps_attempt[~accepted], dLs[~accepted], c="red",
                    marker="x", label="rejected")
    axes[0].axhline(0, color="gray", lw=0.5)
    axes[0].set(xlabel="Epoch", ylabel=r"$\Delta L$", title="Move outcomes")
    axes[0].legend(fontsize=7)

    eps_all = np.array([h["epoch"] for h in history_custom])
    T_metro = np.array([h["radapt_T"] if h["radapt_T"] is not None else np.nan
                        for h in history_custom])
    axes[1].plot(eps_all, T_metro, "-", color="C1")
    axes[1].set(xlabel="Epoch", ylabel=r"$T_{\rm metro}$",
                title="Metropolis temperature", yscale="log")

    bins = np.arange(-0.5, n_custom + 0.5, 1)

    js = np.array([h["radapt_j"] for h in attempts if h["radapt_accepted"]])
    is_ = np.array([h["radapt_i"] for h in attempts if h["radapt_accepted"]])
    axes[2].hist([js, is_], bins=bins, label=["merge j", "split i"], stacked=False)
    axes[2].set(xlabel="Interval index", ylabel="Count",
                title=f"Accepted move participation ({len(js)})")
    axes[2].legend(fontsize=7)

    js_r = np.array([h["radapt_j"] for h in attempts if not h["radapt_accepted"]])
    is_r = np.array([h["radapt_i"] for h in attempts if not h["radapt_accepted"]])
    axes[3].hist([js_r, is_r], bins=bins, label=["merge j", "split i"], stacked=False)
    axes[3].set(xlabel="Interval index", ylabel="Count",
                title=f"Rejected move participation ({len(js_r)})")
    axes[3].legend(fontsize=7)

    fig.suptitle(f"r-adapt diagnostics ({accepted.sum()}/{len(attempts)} accepted)",
                 fontsize=11)
    plt.show()
else:
    print("No r-adapt attempts in history (radapt_enable=False or warmup/cooldown).")

## Timesteps evolution video

In [ ]:
# from IPython.display import Video, Image
# from utils import save_timesteps_video

# video_dir = os.path.join(os.getcwd(), "results", "pann")
# os.makedirs(video_dir, exist_ok=True)
# video_path = save_timesteps_video(video_dir, "custom_timesteps", history=history_custom, T=T)

# (Video(video_path, embed=True) if video_path.endswith(".mp4") else Image(video_path))